# 05 - Methode 2 : t-SNE

IFT599-1 / IFT799-51 / IFT799-52 - TP1

Membres : Nom Prenom - CIP

Objectif : t-SNE genere directement deux dimensions, il n'y a donc pas de
selection de paires. On produit la carte de toutes les classes, puis une
carte par paire de classes.

Source : scikit-learn, sklearn.manifold.TSNE.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("..")

from itertools import combinations

import pandas as pd
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

from src.config import CLASSES, PERPLEXITE, SEED
from src.loading import charger
from src.distances import euclidienne
from src.separability import overlap
from src.plotting import nuage

X, y = charger()
Z = StandardScaler().fit_transform(X)

## Carte de toutes les classes

t-SNE depend de l'initialisation : random_state est fixe pour que les
resultats soient reproductibles.

In [ ]:
T = TSNE(n_components=2, perplexity=PERPLEXITE, random_state=SEED).fit_transform(Z)

nuage(T, y, CLASSES, (0, 1),
      f"t-SNE - toutes les classes (perplexite {PERPLEXITE})",
      "../figures/tsne/globale.png")

## Une carte par paire de classes

t-SNE est recalcule sur les seuls patients des deux classes concernees.

In [ ]:
lignes = []
for a, b in combinations(CLASSES, 2):
    garde = (y == a) | (y == b)
    Tp = TSNE(n_components=2, perplexity=PERPLEXITE, random_state=SEED).fit_transform(Z[garde])
    yp = y[garde]

    o = overlap(Tp[yp == a], Tp[yp == b], euclidienne)
    lignes.append({"paire": f"{a}-{b}", "overlap": round(o, 3)})

    nuage(Tp, yp, [a, b], (0, 1),
          f"t-SNE - {a} vs {b}",
          f"../figures/tsne/{a}_{b}.png")

t = pd.DataFrame(lignes)
t.to_csv("../results/selection_tsne.csv", index=False)
t

## Effet de la perplexite

La diapositive 63 conseille de tester plusieurs perplexites.

In [ ]:
for p in [5, 50]:
    Tp = TSNE(n_components=2, perplexity=p, random_state=SEED).fit_transform(Z)
    nuage(Tp, y, CLASSES, (0, 1),
          f"t-SNE - perplexite {p}",
          f"../figures/tsne/perplexite_{p}.png")

### Commentaires

**Huit paires sur dix** passent sous le seuil, de 0.227 (COAD-KIRC) a 0.680
(COAD-LUAD). La carte globale montre cinq groupes compacts et nettement separes.
C'est la reponse a la question du TP : les cinq types de cancers sont bien
separables.

Les deux exceptions, BRCA-PRAD (1.804) et BRCA-LUAD (1.556), montrent pourtant
des nuages visuellement distincts. Ce sont quelques patients isoles dans la
region de l'autre classe qui font basculer la mesure, la distance intra-classe
etant un maximum.

Les perplexites 5, 30 et 50 conservent toutes les cinq groupes : la structure est
reelle et non un artefact de parametrage. A 5 les groupes se fragmentent, a 50
ils sont plus lisses.

t-SNE (8/10) fait beaucoup mieux que l'ACP (1/10) sur les memes donnees. L'ACP
est lineaire et preserve la variance globale ; t-SNE est non lineaire et preserve
les voisinages locaux. Or deux patients d'un meme cancer sont des voisins
proches, et c'est exactement ce que t-SNE conserve. En contrepartie, ses axes ne
sont pas interpretables et le resultat depend de l'initialisation.

**Synthese**

| Representation | Paires avec Overlap < 1 |
|---|---|
| 750 variables (Methode 1) | 1 / 10 |
| Meilleures paires de genes | 5 / 10 |
| ACP (15 composantes) | 1 / 10 |
| t-SNE | 8 / 10 |

Transformer les donnees n'ameliore donc pas en soi la separation : l'ACP
transforme et degrade, t-SNE transforme et ameliore. Ce qui compte est
l'objectif de la transformation.

Les 750 genes etant tires au hasard parmi 20264, ces resultats constituent une
borne inferieure.